In [ ]:
#=========================================================
# stitch_files.py
# Author: McKenna W. Stanford
# Date Created: 03/17/2023
#=========================================================

In [1]:
#=======================================
# Imports
#=======================================
import numpy as np
import glob
import xarray
import datetime
import pandas as pd
import os

# There are 2 parts to this script
## Part 1:
###  Uses a single function to return a 3D stitched grid variable.
## Part 2:
###   Uses several functions to write 3D output to a new NetCDF file after stitching. 

# The following is independent of parts. You need to list the PLT files and construct a list of "unique output times".

In [3]:
#=======================================
# Get list of PLT files
#=======================================
#sim_name = 'cntl_bulk_ice_ABIFM'
#sim_name = 'cntl_bulk_ice_ABIFM_hm'
#sim_name = 'sip_bulk_ice_ABIFM'
#sim_name = 'sip_bulk_ice_ABIFM_hm'
#sim_name = 'sip_10x_bulk_ice_ABIFM'
sim_name = 'sip_10x_bulk_ice_ABIFM_hm'

path = '/pscratch/sd/m/mckenna/dharma_output/'+sim_name+'/'
#path = '/pscratch/sd/m/mckenna/dharma_output/'+sim_name+'/restart/'
plt_files = sorted(glob.glob(path+'dharma_PLT*'))
num_plt_files = len(plt_files)
print('# of PLT files:',num_plt_files)

# of PLT files: 5328


## The list of "unique_output_times" are needed to index files in both Parts 1 and 2

In [4]:
#=======================================
# Get list of time and processor number
# values from all files using the file
# name's string.
# Then get unique output times and processor
# numbers
#=======================================
output_times = []
proc_numbers = []
for ii in range(num_plt_files):
    file_str = plt_files[ii]
    # Split it up by '/' to get rid of the path
    file_str = file_str.split('/')[-1]
    # Get rid of ".cdf" extention
    file_str = file_str.split('.')[0]
    # Split up by '_' to get output "time" and processor number
    file_str = file_str.split('_')
    tmp_time = file_str[2]
    tmp_proc = file_str[3]
    output_times.append(tmp_time)
    proc_numbers.append(tmp_proc)
output_times = np.array(output_times)
proc_numbers = np.array(proc_numbers)
unique_output_times = np.unique(output_times)
unique_procs = np.unique(proc_numbers)
num_unique_output_times = len(unique_output_times)
num_unique_procs = len(unique_procs)
print('# of unique output times:',num_unique_output_times)
print('# of processors:',num_unique_procs)

# of unique output times: 37
# of processors: 144


***
# Part 1 - return 3D structured grid of desired variable given a desired output time
***

In [5]:
def return_3d_var(key,chop_files,nx,ny,nz,nboxes):
    """
    Provided a key (which is a variable name *without* the processor # extension,
    e.g. 'qc'), as well as the list of chop files, nx, ny, nz, and nboxes (= # of processors),
    return a NumPY array that stitches together all files into a single 3D grid.
    """

    # Initialize empy master array
    if key == 'w':
        vdata = np.zeros((nx,ny,nz+1))
    elif key == 'u':
        vdata = np.zeros((nx+1,ny,nz))
    elif key == 'v':
        vdata = np.zeros((nx,ny+1,nz))
    else:
        vdata = np.zeros((nx,ny,nz))
    

    # Loop through number of processor files
    # Could parallelize this eventually
    for jj in range(nboxes):
        ncfile = xarray.open_dataset(chop_files[jj])
        tmp_keys = list(ncfile.keys())
        tmp_key = tmp_keys[0]
        # Get extension
        extension = tmp_key[-5:]
        
        full_key = key + extension
        tmp_var = ncfile[full_key]
        
        # Get 'bnds' and 'ngrow'
        bnds = tmp_var.attrs['bnds']
        ngrow = tmp_var.attrs['ngrow']

        ib = bnds[0]-1
        jb = bnds[1]-1
        kb = bnds[2]-1
        ie = bnds[3]-1
        je = bnds[4]-1
        ke = bnds[5]-1
        mx = ie-ib+1
        my = je-jb+1
        mz = ke-kb+1

        tmp_var_2 = np.reshape(tmp_var.values,(mx,my,mz),order='F') 
        vdata[ib+ngrow:ie-ngrow+1,jb+ngrow:je-ngrow+1,kb+ngrow:ke-ngrow+1] = tmp_var_2[ngrow:mx-ngrow,ngrow:my-ngrow,ngrow:mz-ngrow]
       
    return vdata

***
# Part 2 - Write stitched 3D grids to a NetCDF file
### Needs some specific modifications for your files. We explicitly define the variables we are writing to NetCDF files in the "write_to_file" function
***

## This part needs the sounding files

In [6]:
#=======================================
# Get Variables from Sounding File
# and add to a dictionary. 
#
#These are used to calculate pressure, theta, and
# temperature in the 3D files and makes
# the heights (zt and zw) as well as
# the reference state density (rhobar)
# accessible.
#=======================================
sounding_file = path + 'dharma.soundings.cdf'
ncfile_sounding = xarray.open_dataset(sounding_file)
zt = ncfile_sounding['zt'].values # m
zw = ncfile_sounding['zw'].values # m
rhobar = ncfile_sounding['rhobar'].values # kg/m^3
sounding_th = ncfile_sounding['th'].values # K
T = ncfile_sounding['T'].values # K
sounding_time = ncfile_sounding['time'].values
theta_00 = ncfile_sounding.attrs['theta_00']
ncfile_sounding.close()


sounding_dict = {'zt':zt,\
                 'zw':zw,\
                 'rhobar':rhobar,\
                 'th':sounding_th,\
                 'T':T,\
                 'time':sounding_time,\
                 'theta_00':theta_00,\
                 }

#============================
# Optional stuff to compute pressure,
# temperature, and theta from only
# the sounding files
#============================

#Rgas = 8314.3
#Mw = 28.966
#cp = 1004.
#Rd = Rgas/Mw
#grav = 9.81
#p_exner = 1.e5

# Compute theta
#theta = (th+1)*theta_00
#pi = T/theta
#penv = (pi**(cp/Rd))*p_exner # in Pa
# Convert to mb
#penv = penv*1.e-2

/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [7]:
def stitch_lvl_2(key,chop_files,nx,ny,nz,nboxes):
    """
    Provided a key (which is a variable name *without* the processor # extension,
    as well as the list of chop files, nx,ny,nz, and nboxes(=# of processors),
    # return a NumPY array that stitches together all files into a single 3D grid.
    """

    # Initialize empy master array
    if key == 'w':
        vdata = np.zeros((nx,ny,nz+1))
    elif key == 'u':
        vdata = np.zeros((nx+1,ny,nz))
    elif key == 'v':
        vdata = np.zeros((nx,ny+1,nz))
    else:
        vdata = np.zeros((nx,ny,nz))
    

    # Loop through number of processor files
    # Could parallelize this eventually

    # Loop through chops
    for jj in range(nboxes):
        ncfile = xarray.open_dataset(chop_files[jj])
        tmp_keys = list(ncfile.keys())
        tmp_key = tmp_keys[0]
        # Get extension
        extension = tmp_key[-5:]
        
        full_key = key + extension
        tmp_var = ncfile[full_key]
        
        # Get 'bnds' and 'ngrow'
        bnds = tmp_var.attrs['bnds']
        ngrow = tmp_var.attrs['ngrow']

        ib = bnds[0]-1
        jb = bnds[1]-1
        kb = bnds[2]-1
        ie = bnds[3]-1
        je = bnds[4]-1
        ke = bnds[5]-1
        mx = ie-ib+1
        my = je-jb+1
        mz = ke-kb+1

        tmp_var_2 = np.reshape(tmp_var.values,(mx,my,mz),order='F') 
        vdata[ib+ngrow:ie-ngrow+1,jb+ngrow:je-ngrow+1,kb+ngrow:ke-ngrow+1] = tmp_var_2[ngrow:mx-ngrow,ngrow:my-ngrow,ngrow:mz-ngrow]
       
    return vdata

In [8]:
def stitch_lvl_1(in_output_time,plt_file_list,alt_file_list,output_times,sounding_dict,sim_name,interp_winds_flag):

        """
        Given the list of PLT files, the list of output times, and the
        desired output time to process, this will find all the
        files associated with the desired output time, then read in the
        first file to grab some metadata.
        Then send it to "stitch_lvl_2" function to send back a stitched
        numpy array.
        """

        print('Stitching and writing file for output time {} for simulation named {}'.format(in_output_time,sim_name))
        
        file_ids = np.where(output_times == in_output_time)
        chop_files = np.array(plt_file_list)[file_ids]
        num_chop_files = len(chop_files)
        if alt_file_list is not None:
            alt_chop_files = np.array(alt_file_list)[file_ids]
        # Read in the first file to get metadata

        ncfile = xarray.open_dataset(chop_files[0])
        nboxes = ncfile.attrs['nboxes'] # number of processors
        pmap = ncfile.attrs['pmap'] 
        time = ncfile.attrs['time'] # This is the actual output time in seconds
        nx = ncfile.attrs['nx'] # number of grid points in "x" direction
        ny = ncfile.attrs['ny'] # number of grid points in "y" direction
        nz = ncfile.attrs['nz'] # number of height levels
        L_x = ncfile.attrs['L_x'] # length of "x" direction in m
        L_y = ncfile.attrs['L_y'] # length of "y" direction in m
        H = ncfile.attrs['H'] # Highest height in domain
        ncfile.close()
        
        
        # compute x and y coordinates of grid-point centers
        # and horizontal grid spacing
        dx = L_x/nx # grid spacing
        x = dx/2. + np.arange(0,nx,1)*dx
        dy = L_y /ny # grid spacing
        y  = dy/2. + np.arange(0,ny)*dy
        # Grid point centers for u and v staggered grids
        xu = np.arange(0,nx+1)*dx
        yv = np.arange(0,ny+1)*dy  
        
        # Get variables names
        keys = list(ncfile.keys())
        num_keys = len(keys)
        # For development we will loop through the keys
        # and stitch them with "stitch_lvl_2"
        # Hope to make this parallel at some point.
        

        
        var_dict = {}
        # Send in a key that is trimmed from the processor number appendage
        for key in keys:
            key_trim = key[0:-5]
            #print('Stitching key:',key_trim)
            return_var = stitch_lvl_2(key_trim,chop_files,nx,ny,nz,nboxes)
            var_dict[key_trim] = return_var
            
        # Now do the alt files
        if alt_file_list is not None:
            key_trim = 'reff_c'
            return_var = stitch_lvl_2(key_trim,alt_chop_files,nx,ny,nz,nboxes)
            var_dict[key_trim] = return_var    
            
            
        # To calculate 3D theta, temperature, and pressure,
        # need variables from the sounding file, which has a
        # time dimension, so need to find the time ID of the
        # sounding that matches the time of the current file
        #
        # Send that time id to the function
        time_id = np.where(sounding_dict['time'] == time)[0][0]
                    
        state_dict = calc_pressure_theta(var_dict,sounding_dict,time_id)
        for key,val in state_dict.items():
            var_dict[key] = val
            
        # Add heights and mean density to dictionary
        var_dict['zt'] = sounding_dict['zt']
        var_dict['zw'] = sounding_dict['zw']
        var_dict['rhobar'] = sounding_dict['rhobar'][time_id,:]
        
    
        if interp_winds_flag:
            u_interp,v_interp,w_interp = interp_winds(var_dict['u'],var_dict['v'],var_dict['w'],x,xu,y,yv,var_dict['zt'],var_dict['zw'])
            
            var_dict['u_interp'] = u_interp
            var_dict['v_interp'] = v_interp
            var_dict['w_interp'] = w_interp
        
        
        var_dict['time'] = time
        # Write to file
        write_to_file(var_dict,sim_name,x,xu,y,yv,var_dict['zt'],var_dict['zw'],in_output_time)
        
        return var_dict

In [9]:
def interp_winds(u,v,w,x,xu,y,yv,zt,zw):
    
    from scipy.interpolate import RegularGridInterpolator as rgi
    xi,yi,zti = np.meshgrid(x,y,zt,indexing='ij')

    w_interp_func = rgi((x,y,zw), w)
    w_interp = w_interp_func(np.array([xi,yi,zti]).T).T
    
    u_interp_func = rgi((xu,y,zt), u)
    u_interp = u_interp_func(np.array([xi,yi,zti]).T).T
    
    v_interp_func = rgi((x,yv,zt), v)
    v_interp = v_interp_func(np.array([xi,yi,zti]).T).T
    
    return u_interp,v_interp,w_interp

In [10]:
def calc_pressure_theta(var_dict,sounding_dict,time_id):
    """
    Calculate 3D pressure, potential temperature, and temperature, which requires
    pulling some variables from the sounding file
    """
    # Constants
    Rgas = 8314.3
    Mw = 28.966
    cp = 1004.
    Rd = Rgas/Mw
    grav = 9.81
    p_exner = 1.e5
        
    state_dict = {}
    
    theta_3d = (var_dict['th'] + 1)*sounding_dict['theta_00']
    theta_sound = (sounding_dict['th'] + 1)*sounding_dict['theta_00']
    nz = len(sounding_dict['zt'])
    nx = len(var_dict['th'][:,0,0])
    ny = len(var_dict['th'][0,:,0])     

    
    
    pi_sound = np.zeros(nz)
    for kk in range(nz):
        pi_sound[kk] = sounding_dict['T'][time_id,kk]/theta_sound[time_id,kk]
        
    T_3d = pi_sound*theta_3d 
    
    pi_3d = T_3d/theta_3d
    penv_3d = (pi_3d**(cp/Rd))*p_exner # in Pa

    
    state_dict['pressure'] = penv_3d
    state_dict['temperature'] = T_3d
    state_dict['theta'] = theta_3d
    state_dict['exner'] = pi_3d
    
    return state_dict

In [11]:
def write_to_file(var_dict,sim_name,x,xu,y,yv,zt,zw,in_output_time):

    
    data_vars = dict(
            pressure=(["nx","ny","nzt"],var_dict["pressure"],{"units":"Pa","long_name":"Atmospheric pressure"}),
            theta=(["nx","ny","nzt"],var_dict["theta"],{"units":"K","long_name":"Atmospheric potential temperature"}),
            temperature=(["nx","ny","nzt"],var_dict["temperature"],{"units":"K","long_name":"Atmospheric temperature"}),
            u=(["nxu","ny","nzt"],var_dict["u"],{"units":"m/s","long_name":"Zonal velocity on native staggered grid"}),
            u_interp=(["nx","ny","nzt"],var_dict["u_interp"],{"units":"m/s","long_name":"Zonal velocity interpolated on regular grid"}),
            v=(["nx","nyv","nzt"],var_dict["v"],{"units":"m/s","long_name":"Meridional velocity on native staggered grid"}),
            v_interp=(["nx","ny","nzt"],var_dict["v_interp"],{"units":"m/s","long_name":"Meridional velocity interpolated on regular grid"}),
            w=(["nx","ny","nzw"],var_dict["w"],{"units":"m/s","long_name":"Vertical velocity on native staggered grid"}),
            w_interp=(["nx","ny","nzt"],var_dict["w_interp"],{"units":"m/s","long_name":"Vertical velocity on interpolated on regular grid"}),
            qv=(["nx","ny","nzt"],var_dict["qv"],{"units":"kg/kg","long_name":"Water vapor mixing ratio"}),
            qc=(["nx","ny","nzt"],var_dict["qc"],{"units":"kg/kg","long_name":"Cloud water mass mixing ratio"}),
            nc=(["nx","ny","nzt"],var_dict["nc"],{"units":"/kg","long_name":"Cloud water number concentration mixing ratio"}),
            qr=(["nx","ny","nzt"],var_dict["qr"],{"units":"kg/kg","long_name":"Rain mass mixing ratio"}),
            nr=(["nx","ny","nzt"],var_dict["nr"],{"units":"/kg","long_name":"Rain number concentration mixing ratio"}),
            qic=(["nx","ny","nzt"],var_dict["qic"],{"units":"kg/kg","long_name":"Cloud Ice mass mixing ratio"}),
            nic=(["nx","ny","nzt"],var_dict["nic"],{"units":"/kg","long_name":"Cloud Ice number concentration mixing ratio"}),
            qif=(["nx","ny","nzt"],var_dict["qif"],{"units":"kg/kg","long_name":"Fluffy ice mass mixing ratio"}),
            nif=(["nx","ny","nzt"],var_dict["nif"],{"units":"/kg","long_name":"Fluffy ice number concentration mixing ratio"}),
            qid=(["nx","ny","nzt"],var_dict["qid"],{"units":"kg/kg","long_name":"Dense ice mass mixing ratio"}),
            nid=(["nx","ny","nzt"],var_dict["nid"],{"units":"/kg","long_name":"Dense ice number concentration mixing ratio"}),
            sat=(["nx","ny","nzt"],var_dict["sat"],{"units":"-","long_name":"Supersaturation w.r.t. liquid water"}),
            na_1=(["nx","ny","nzt"],var_dict["na_1"],{"units":"/kg","long_name":"Aerosol mode 1 aersol number concentration"}),
            nac_1=(["nx","ny","nzt"],var_dict["nac_1"],{"units":"/kg","long_name":"Aerosol mode 1 activated aersol number concentration"}),
            na_2=(["nx","ny","nzt"],var_dict["na_2"],{"units":"/kg","long_name":"Aerosol mode 2 aersol number concentration"}),
            nac_2=(["nx","ny","nzt"],var_dict["nac_2"],{"units":"/kg","long_name":"Aerosol mode 2 activated aersol number concentration"}),
            na_3=(["nx","ny","nzt"],var_dict["na_3"],{"units":"/kg","long_name":"Aerosol mode 3 aersol number concentration"}),
            nac_3=(["nx","ny","nzt"],var_dict["nac_3"],{"units":"/kg","long_name":"Aerosol mode 3 activated aersol number concentration"}),
            rhobar=(["nzt"],var_dict["rhobar"],{"units":"kg/m^3","long_name":"Reference air density"}),
    )


    coords=dict(x=(["nx"],x,{"units":"meters","long_name":"Grid point centers x direction for state variable grid"}),\
        xu=(["nxu"],xu,{"units":"meters","long_name":"Grid point centers for staggered zonal velocity grid"}),\
        y=(["ny"],y,{"units":"meters","long_name":"Grid point centers for state variable grid"}),\
        yu=(["nyv"],yv,{"units":"meters","long_name":"Grid point centers for staggered meridional velocitygrid"}),\
        zt=(["nzt"],var_dict['zt'],{"units":"meters","long_name":"Height for state variable grid"}),\
        zw=(["nzw"],var_dict['zw'],{"units":"meters","long_name":"Height for staggered vertical velocity grid"}),\
        time=(["nt"],[var_dict['time']],{"units":"Seconds since start of simulation","long_name":"Time"}),\
    )

    current_time = datetime.datetime.now()
    current_time = current_time.strftime('%m/%d/%Y %H:%M EST')
    attrs={'creation_date':current_time, 
             'author':'McKenna W. Stanford', 
             'email':'mws2175@columbia.edu'}
    ds = xarray.Dataset(data_vars,coords,attrs)
    save_path = '/pscratch/sd/m/mckenna/dharma_3d/'+sim_name+'/'
    nc_file_name = 'dharma_3d_{}_{}.nc'.format(sim_name,in_output_time)
    ds.to_netcdf(save_path+nc_file_name, mode='w')   


## Run the code here

In [12]:
#======================================================
# Send an output time to the "stitch" function, along with the *total*
# list of PLT files and the *total* list of output times. 
# The function will find the correct files to stitch
# together for a given output time. Note that "output time" means the
# output time in the string, which is the desired output time (e.g., hourly, or
# 3600 seconds) divided by the time step. So, if your output frequency is 1 hour
# and your time step is 2 seconds, the output time would be 3600/2 = 1800.
#======================================================
# Define your simulation name, too.
for dum_ii in range(len(unique_output_times)):
    #stitch_lvl_1(unique_output_times[dum_ii],plt_files,alt_files,output_times,sounding_dict,sim_name='cntl_sb_1sec',interp_winds_flag=True)    
    var_dict = stitch_lvl_1(unique_output_times[dum_ii],plt_files,None,output_times,sounding_dict,sim_name=sim_name,interp_winds_flag=True)    
    #print(aaaaa)

Stitching and writing file for output time 021600 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 022200 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 022800 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 023400 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 024000 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 024600 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 025200 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 025800 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 026400 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for output time 027000 for simulation named sip_10x_bulk_ice_ABIFM_hm
Stitching and writing file for

In [13]:
print('done')

done


In [14]:
unique_output_times

array(['021600', '022200', '022800', '023400', '024000', '024600',
       '025200', '025800', '026400', '027000', '027600', '028200',
       '028800', '029400', '030000', '030600', '031200', '031800',
       '032400', '033000', '033600', '034200', '034800', '035400',
       '036000', '036600', '037200', '037800', '038400', '039000',
       '039600', '040200', '040800', '041400', '042000', '042600',
       '043200'], dtype='<U6')

### Diagnostics: Read in file to check dataset

In [ ]:

# Test reading in file and checking values
read_path = '/mnt/raid/mwstanfo/dharma_3d/'
file_name = 'dharma_3d_bulk_3600.nc'
ncfile = xarray.open_dataset(read_path+file_name,decode_times=False)
ncfile